# Complaint Copilot — Part 3: Emerging Issue Clusters

Groups complaint narratives into **topics** over the embeddings, then flags which clusters are
**emerging** (growing in the most recent window). Uses **BERTopic** when available and falls back
to KMeans so it always produces clusters. Saves a summary the Streamlit dashboard reads.

Inputs: `data/complaints.parquet` (from notebook 01) — needs `date_received` and `narrative`.
Falls back to a small dated sample if the parquet is missing.

In [ ]:
# Run once if needed:
# %pip install bertopic umap-learn hdbscan scikit-learn sentence-transformers pandas pyarrow

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_PATH = os.path.join("data", "complaints.parquet")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RECENT_DAYS = 30          # window that counts as "recent" for emergence
os.makedirs("data", exist_ok=True)

In [ ]:
def _sample_df():
    import pandas as pd
    rows = [
        {"complaint_id":"S-0001","date_received":"2024-03-22","product":"Credit reporting","issue":"Improper use of your report","narrative":"There are several hard inquiries on my report that I never authorized. I think someone is using my identity to open accounts."},
        {"complaint_id":"S-0002","date_received":"2024-03-20","product":"Credit reporting","issue":"Incorrect information on your report","narrative":"I reported a fraudulent account three times but the bureau keeps saying it is verified and never explains how. This account is not mine."},
        {"complaint_id":"S-0003","date_received":"2024-01-05","product":"Debt collection","issue":"Attempts to collect debt not owed","narrative":"The collector keeps calling about a debt I already paid and never closed the file. They add fees that were never explained."},
        {"complaint_id":"S-0004","date_received":"2024-01-08","product":"Debt collection","issue":"Communication tactics","narrative":"No response from the company for over a month after I disputed the amount owed, then repeated calls every day."},
        {"complaint_id":"S-0005","date_received":"2024-02-01","product":"Mortgage","issue":"Trouble during payment process","narrative":"My mortgage servicer changed my escrow and doubled my payment without any notice and cannot explain the new amount."},
        {"complaint_id":"S-0006","date_received":"2024-02-03","product":"Mortgage","issue":"Applying for a mortgage","narrative":"The loan officer lost my documents twice and the closing was delayed by weeks, which almost cost me the house."},
        {"complaint_id":"S-0007","date_received":"2024-03-15","product":"Money transfer","issue":"Fraud or scam","narrative":"I sent money to the wrong person through the app and support says nothing can be done. There is no way to reverse the transfer."},
        {"complaint_id":"S-0008","date_received":"2024-03-16","product":"Money transfer","issue":"Other transaction problem","narrative":"The mobile payment transfer was marked complete but the recipient never received it. The funds are just gone."},
        {"complaint_id":"S-0009","date_received":"2024-03-21","product":"Money transfer","issue":"Other transaction problem","narrative":"Digital wallet transfer reversal sent to the wrong recipient with no clear way to recover the money."},
        {"complaint_id":"S-0010","date_received":"2024-03-19","product":"Money transfer","issue":"Fraud or scam","narrative":"Scam payment through the wallet app, money sent to a stranger and the app will not help reverse it."},
        {"complaint_id":"S-0011","date_received":"2024-01-12","product":"Credit card","issue":"Fees or interest","narrative":"They charged me interest even though I paid the full statement balance before the due date and it keeps compounding."},
        {"complaint_id":"S-0012","date_received":"2024-02-20","product":"Credit card","issue":"Problem with a purchase shown on statement","narrative":"I was charged twice for the same purchase and the billing dispute was denied without any real review."},
        {"complaint_id":"S-0013","date_received":"2024-03-18","product":"Credit reporting","issue":"Incorrect information on your report","narrative":"A paid collection still shows as open after 60 days even though I sent the settlement confirmation letter."},
        {"complaint_id":"S-0014","date_received":"2024-01-25","product":"Student loan","issue":"Dealing with your lender or servicer","narrative":"My payments are not applied to principal and the servicer keeps putting me in forbearance I never requested."},
    ]
    return pd.DataFrame(rows)

if os.path.exists(DATA_PATH):
    df = pd.read_parquet(DATA_PATH)
    if "date_received" not in df.columns:
        print("parquet has no date_received; using dated sample instead.")
        df = _sample_df()
    else:
        print("Loaded", len(df), "complaints from", DATA_PATH)
else:
    df = _sample_df()
    print("No parquet found; using dated sample of", len(df), "complaints.")
df = df.dropna(subset=["narrative"]).reset_index(drop=True)
df.head(2)

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)
docs = df["narrative"].astype(str).tolist()
emb = np.asarray(embedder.encode(docs, normalize_embeddings=True, show_progress_bar=True), dtype="float32")
print("embedded", emb.shape)

## Fit topics
BERTopic first; if it isn't installed or fails on small data, fall back to KMeans with
TF-IDF labels. Either way you get a `topic` and a readable `label` per complaint.

In [ ]:
def fit_topics(docs, emb):
    n = len(docs)
    try:
        from bertopic import BERTopic
        from umap import UMAP
        from hdbscan import HDBSCAN
        umap_model = UMAP(n_neighbors=min(15, max(2, n - 1)),
                          n_components=min(5, max(2, n - 2)),
                          min_dist=0.0, metric="cosine", random_state=42)
        hdbscan_model = HDBSCAN(min_cluster_size=max(2, n // 10),
                                metric="euclidean", cluster_selection_method="eom",
                                prediction_data=True)
        tm = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model,
                      calculate_probabilities=False, verbose=False)
        topics, _ = tm.fit_transform(docs, embeddings=emb)
        info = tm.get_topic_info()
        name = dict(zip(info["Topic"], info["Name"]))
        labels = [name.get(t, str(t)) for t in topics]
        return list(topics), labels, "bertopic"
    except Exception as e:
        print("BERTopic unavailable/failed (", e, ") -- using KMeans fallback.")
        from sklearn.cluster import KMeans
        from sklearn.feature_extraction.text import TfidfVectorizer
        k = max(2, min(6, n // 3 or 2))
        topics = KMeans(n_clusters=k, n_init=10, random_state=42).fit(emb).labels_.tolist()
        vec = TfidfVectorizer(stop_words="english", max_features=2000)
        X = vec.fit_transform(docs)
        terms = np.array(vec.get_feature_names_out())
        lab = {}
        for c in range(k):
            ridx = [i for i, t in enumerate(topics) if t == c]
            centroid = np.asarray(X[ridx].mean(axis=0)).ravel()
            lab[c] = ", ".join(terms[np.argsort(-centroid)[:3]])
        labels = [lab.get(t, str(t)) for t in topics]
        return topics, labels, "kmeans"

topics, labels, method = fit_topics(docs, emb)
df["topic"] = topics
df["label"] = labels
print("method:", method, "| topics found:", len(set(topics)))
df[["complaint_id", "product", "topic", "label"]].head(8)

## Detect emerging clusters
Compare each topic's volume in the most recent window against everything before it.
Positive growth = an issue that is rising.

In [ ]:
def topic_trends(df, recent_days=RECENT_DAYS):
    d = df.copy()
    d["date"] = pd.to_datetime(d["date_received"], errors="coerce")
    d = d.dropna(subset=["date"])
    if d.empty:
        # no usable dates: just report sizes
        out = (df.groupby(["topic", "label"]).size()
                 .reset_index(name="recent"))
        out["prior"] = 0
        out["growth"] = out["recent"]
        return out.sort_values("growth", ascending=False)
    cutoff = d["date"].max() - pd.Timedelta(days=recent_days)
    rec = d[d["date"] > cutoff]["topic"].value_counts()
    pri = d[d["date"] <= cutoff]["topic"].value_counts()
    rows = []
    for t in sorted(d["topic"].unique()):
        r, p = int(rec.get(t, 0)), int(pri.get(t, 0))
        lbl = d[d["topic"] == t]["label"].iloc[0]
        rows.append({"topic": t, "label": lbl, "recent": r, "prior": p,
                     "size": r + p, "growth": r - p})
    return pd.DataFrame(rows).sort_values("growth", ascending=False).reset_index(drop=True)

trends = topic_trends(df)
trends

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sizes = trends.sort_values("size")
ax[0].barh(sizes["label"].astype(str).str.slice(0, 30), sizes["size"])
ax[0].set_title("Cluster size")
rising = trends.sort_values("growth")
colors = ["#c0392b" if g > 0 else "#7f8c8d" for g in rising["growth"]]
ax[1].barh(rising["label"].astype(str).str.slice(0, 30), rising["growth"], color=colors)
ax[1].set_title("Recent-vs-prior growth (emerging)")
plt.tight_layout(); plt.show()

In [ ]:
# Save for the dashboard
trends.to_parquet("data/topic_summary.parquet", index=False)
df[["complaint_id", "topic", "label"]].to_parquet("data/complaints_topics.parquet", index=False)
print("Saved data/topic_summary.parquet and data/complaints_topics.parquet")
print("\nTop emerging clusters:")
print(trends.head(3)[["label", "recent", "prior", "growth"]].to_string(index=False))

The Streamlit dashboard (`app.py`) automatically shows an **Emerging issue clusters** panel
when `data/topic_summary.parquet` exists. Re-run this notebook to refresh it.

**Next:** for real data this gets much richer — try BERTopic's `topics_over_time` for a proper
time series per topic, and label clusters with an LLM for human-readable names.